# Setup

We consider a short position in 100 European put options, delta-hedged
using the underlying stock.

The stock price follows a geometric Brownian motion under the physical
probability measure:

$$
\frac{dS_t}{S_t}=\mu\,dt+\sigma\,dW_t.
$$

The portfolio value is:

$$
V_t=h_tS_t+Y_t-P^{BS}(t,S_t),
$$

where $h_t$ is the put delta and $Y_t$ is the cash position.

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

S0 = 152.51
K = 170
T = 0.292

r = 0.00119
sigma = 0.4907
mu = 0.1691

dt = 1 / 252
days = 10

M = 100
N = 40_000
alpha = 0.95

In [5]:
def put_price(S, tau):

    d1 = (
        np.log(S / K) +
        (r + 0.5 * sigma**2) * tau
    ) / (sigma * np.sqrt(tau))

    d2 = d1 - sigma * np.sqrt(tau)

    return (
        K * np.exp(-r * tau) * norm.cdf(-d2)
        - S * norm.cdf(-d1)
    )


def put_delta(S, tau):

    d1 = (
        np.log(S / K) +
        (r + 0.5 * sigma**2) * tau
    ) / (sigma * np.sqrt(tau))

    return norm.cdf(d1) - 1

In [7]:
P0 = put_price(S0, T)

h0 = put_delta(S0, T)

cash0 = -h0 * S0

V0 = h0 * S0 + cash0 - P0

print("Initial put price:", round(P0, 4))
print("Initial put delta:", round(h0, 4))
print("Initial portfolio value:", round(M * V0, 2))

Initial put price: 27.1077
Initial put delta: -0.6086
Initial portfolio value: -2710.77


# 1

**Pseudocode**

1. Simulate 40,000 stock prices one trading day ahead using
   geometric Brownian motion under the physical measure.

2. Keep the initial stock hedge fixed during the day.

3. Grow the cash position at the risk-free rate.

4. Reprice the put with one day less to maturity.

5. Calculate the portfolio value at the end of the day.

6. Compute the loss for the short position in 100 puts.

7. Estimate the one-day VaR as the 95th percentile of losses.

8. Multiply the one-day VaR by sqrt(10) to estimate
   the ten-day VaR.

In [10]:
np.random.seed(42)

# Simulate one-day stock prices
Z = np.random.normal(0, 1, N)

S1 = S0 * np.exp(
    (mu - 0.5 * sigma**2) * dt
    + sigma * np.sqrt(dt) * Z
)

# Reprice the put
P1 = put_price(S1, T - dt)

# Update cash account
cash1 = cash0 * np.exp(r * dt)

# Final portfolio value
V1 = h0 * S1 + cash1 - P1

# Loss distribution
loss_1day = M * (V0 - V1)

# VaR calculations
var_1day = np.quantile(loss_1day, alpha)

var_10day_scaled = np.sqrt(days) * var_1day

print("One-day VaR: $", round(var_1day, 2))
print("Scaled ten-day VaR: $", round(var_10day_scaled, 2))

One-day VaR: $ 29.89
Scaled ten-day VaR: $ 94.53


# 2

**Pseudocode**

1. Initialize 40,000 simulated portfolios using the initial
   stock price, hedge, and cash position.

2. For each of the ten trading days:

   - Simulate the next stock price.
   - Grow the cash account at the risk-free rate.
   - Reprice the put using the remaining maturity.
   - Calculate the new put delta.
   - Rebalance the stock hedge.
   - Finance the hedge adjustment through the cash account.

3. Calculate the final portfolio value after ten days.

4. Compute the ten-day loss for each simulated portfolio.

5. Estimate the ten-day VaR as the 95th percentile
   of the loss distribution.

In [13]:
np.random.seed(43)

# Initialize portfolios
S = np.full(N, S0)

h = np.full(N, h0)

cash = np.full(N, cash0)

# Simulate ten trading days
for day in range(1, days + 1):

    Z = np.random.normal(0, 1, N)

    # Simulate stock price
    S = S * np.exp(
        (mu - 0.5 * sigma**2) * dt
        + sigma * np.sqrt(dt) * Z
    )

    # Grow cash account
    cash = cash * np.exp(r * dt)

    # Reprice put
    tau = T - day * dt

    P = put_price(S, tau)

    # Rebalance delta hedge
    if day < days:

        h_new = put_delta(S, tau)

        stock_change = h_new - h

        cash = cash - stock_change * S

        h = h_new

# Final portfolio value
V10 = h * S + cash - P

# Ten-day losses
loss_10day = M * (V0 - V10)

# Ten-day VaR
var_10day_simulated = np.quantile(loss_10day, alpha)

print("Simulated ten-day VaR: $", round(var_10day_simulated, 2))

Simulated ten-day VaR: $ 88.71


# 3

In [15]:
results = pd.DataFrame({
    "Method": [
        "One-day Monte Carlo",
        "Ten-day sqrt scaling",
        "Ten-day direct simulation"
    ],
    "95% VaR ($)": [
        var_1day,
        var_10day_scaled,
        var_10day_simulated
    ]
})

display(results.round(2))

difference = var_10day_scaled - var_10day_simulated

percentage = difference / var_10day_simulated * 100

print("Absolute difference: $", round(difference, 2))
print("Percentage difference:", round(percentage, 2), "%")

,Method,95% VaR ($)
0,One-day Monte Carlo,29.89
1,Ten-day sqrt scaling,94.53
2,Ten-day direct simulation,88.71


Absolute difference: $ 5.82
Percentage difference: 6.56 %


**Discussion**

The square-root-of-time method gives a ten-day VaR of $94.53,
while the direct simulation gives $88.71.

The square-root-of-time estimate is therefore approximately
6.6% higher.

The difference arises because square-root scaling assumes that
risk scales proportionally to the square root of time, which is
only an approximation for an option portfolio.

In particular, the direct simulation captures the nonlinear
behavior of the option, changes in delta and gamma exposure,
and the effects of daily delta rebalancing.

Therefore, the direct simulation provides a more detailed
representation of the portfolio's risk over the ten-day horizon,
under the assumed model and hedging strategy.